# 6. STAC Transactions with Authentication

This notebook exercises the [STAC Transactions extension](https://github.com/stac-api-extensions/transactions) against the workshop STAC API at `http://localhost:8084` (stac-auth-proxy).

- **Reads** (GET collections, items, search) are public
- **Writes** (POST/PUT/DELETE) require a bearer token from the mock OIDC server

Each code cell uses assertions so this notebook can also serve as an integration test (`./scripts/run-notebooks.sh`).

> **Note:** Requires the docker-compose auth stack (`MOCK_OIDC_ENDPOINT`). Not available on hosted JupyterHub deployments.

In [ ]:
import json
import time

import httpx

from stac_auth import auth_headers, get_mock_oidc_token, require_local_auth_stack

stac_api_endpoint, mock_oidc_endpoint = require_local_auth_stack()
write_token = get_mock_oidc_token()
write_headers = auth_headers(write_token)

run_id = int(time.time())
collection_id = f"tx-workshop-{run_id}"
item_id = f"tx-item-{run_id}"

print(f"STAC API: {stac_api_endpoint}")
print(f"Mock OIDC: {mock_oidc_endpoint}")
print(f"Test collection: {collection_id}")

## 6.1 Confirm the transactions extension is enabled

In [ ]:
conformance = httpx.get(f"{stac_api_endpoint}/conformance", timeout=10).json()
transaction_classes = [
    uri
    for uri in conformance["conformsTo"]
    if "transaction" in uri.lower()
]

print(json.dumps(transaction_classes, indent=2))
assert transaction_classes, "Expected STAC transaction conformance classes"

## 6.2 Collection transactions

Unauthenticated writes must be rejected. Authenticated writes succeed.

In [ ]:
collection = {
    "id": collection_id,
    "type": "Collection",
    "stac_version": "1.0.0",
    "description": "Temporary collection for transaction/auth tests.",
    "license": "CC-BY-4.0",
    "extent": {
        "spatial": {"bbox": [[-10, -10, 10, 10]]},
        "temporal": {"interval": [["2024-01-01T00:00:00Z", None]]},
    },
    "links": [],
}

denied = httpx.post(
    f"{stac_api_endpoint}/collections",
    json=collection,
    timeout=10,
)
print(f"POST /collections without token: {denied.status_code}")
assert denied.status_code in (401, 403)

created = httpx.post(
    f"{stac_api_endpoint}/collections",
    headers=write_headers,
    json=collection,
    timeout=10,
)
print(f"POST /collections with token: {created.status_code}")
assert created.status_code in (200, 201), created.text

fetched = httpx.get(f"{stac_api_endpoint}/collections/{collection_id}", timeout=10)
print(f"GET /collections/{{id}} (public read): {fetched.status_code}")
assert fetched.status_code == 200
assert fetched.json()["id"] == collection_id

## 6.3 Item transactions

Exercise POST, PUT, and DELETE on items. PUT uses the GET-then-modify pattern recommended by the transactions extension.

In [ ]:
item = {
    "id": item_id,
    "type": "Feature",
    "stac_version": "1.0.0",
    "collection": collection_id,
    "geometry": {"type": "Point", "coordinates": [0.5, 0.5]},
    "bbox": [0.5, 0.5, 0.5, 0.5],
    "properties": {"datetime": "2024-06-01T00:00:00Z"},
    "links": [],
    "assets": {},
}

denied_item = httpx.post(
    f"{stac_api_endpoint}/collections/{collection_id}/items",
    json=item,
    timeout=10,
)
print(f"POST item without token: {denied_item.status_code}")
assert denied_item.status_code in (401, 403)

created_item = httpx.post(
    f"{stac_api_endpoint}/collections/{collection_id}/items",
    headers=write_headers,
    json=item,
    timeout=10,
)
print(f"POST item with token: {created_item.status_code}")
assert created_item.status_code in (200, 201), created_item.text

stored_item = httpx.get(
    f"{stac_api_endpoint}/collections/{collection_id}/items/{item_id}",
    timeout=10,
).json()
stored_item["properties"]["description"] = "Updated by authenticated PUT"

denied_put = httpx.put(
    f"{stac_api_endpoint}/collections/{collection_id}/items/{item_id}",
    json=stored_item,
    timeout=10,
)
print(f"PUT item without token: {denied_put.status_code}")
assert denied_put.status_code in (401, 403)

updated_item = httpx.put(
    f"{stac_api_endpoint}/collections/{collection_id}/items/{item_id}",
    headers=write_headers,
    json=stored_item,
    timeout=10,
)
print(f"PUT item with token: {updated_item.status_code}")
assert updated_item.status_code == 200, updated_item.text
assert (
    updated_item.json()["properties"]["description"]
    == "Updated by authenticated PUT"
)

denied_delete = httpx.delete(
    f"{stac_api_endpoint}/collections/{collection_id}/items/{item_id}",
    timeout=10,
)
print(f"DELETE item without token: {denied_delete.status_code}")
assert denied_delete.status_code in (401, 403)

deleted_item = httpx.delete(
    f"{stac_api_endpoint}/collections/{collection_id}/items/{item_id}",
    headers=write_headers,
    timeout=10,
)
print(f"DELETE item with token: {deleted_item.status_code}")
assert deleted_item.status_code in (200, 204)

missing_item = httpx.get(
    f"{stac_api_endpoint}/collections/{collection_id}/items/{item_id}",
    timeout=10,
)
print(f"GET deleted item: {missing_item.status_code}")
assert missing_item.status_code == 404

## 6.4 Cleanup

Delete the temporary collection. This also requires authentication.

In [ ]:
denied_collection_delete = httpx.delete(
    f"{stac_api_endpoint}/collections/{collection_id}",
    timeout=10,
)
print(f"DELETE collection without token: {denied_collection_delete.status_code}")
assert denied_collection_delete.status_code in (401, 403)

deleted_collection = httpx.delete(
    f"{stac_api_endpoint}/collections/{collection_id}",
    headers=write_headers,
    timeout=10,
)
print(f"DELETE collection with token: {deleted_collection.status_code}")
assert deleted_collection.status_code in (200, 204)

print("\nAll transaction + auth checks passed.")